# Genus two

Regroup the fixed-curve counts of Proposition 3.4 by the quartic point count.

Numerical values use the nonmonic model sum in Lemma 4.5, from [Bergström, *Equivariant counts of points of the moduli spaces of pointed hyperelliptic curves*, §3, equation (3.2)](https://arxiv.org/pdf/math/0611813).

Set $n=\#C(\mathbf F_q)$, $n_2=\#C(\mathbf F_{q^2})$, and $w=\#\operatorname{Weier}(C)(\mathbf F_q)$. The split-node, cusp, and nonsplit-node shifts are $d=1,0,-1$. For a quartic with $N$ rational points, substitute $n=N+d$.

In [1]:
R.<q,n,n2,w,N> = PolynomialRing(QQ)
profiles = [
 ('split node',1,(n*(n-2)+w)/2),
 ('cusp',0,n-w),
 ('nonsplit node',-1,(n2+w-2*q-2)/2),
]
shifted = []
for label,d,multiplicity in profiles:
    value = multiplicity.subs({n:N+d})
    trace = q+1-N-d
    shifted.append((label,d,trace,value))
print(table([(label,N+d,trace,value) for label,d,trace,value in shifted],
           header_row=['profile','normalization points','trace','series on fixed C']))

  profile         normalization points   trace       series on fixed C
├───────────────┼──────────────────────┼───────────┼─────────────────────────┤
  split node      N + 1                  q - N       1/2*N^2 + 1/2*w - 1/2
  cusp            N                      q - N + 1   -w + N
  nonsplit node   N - 1                  q - N + 2   -q + 1/2*n2 + 1/2*w - 1


The weighted sums of $1$, $n_2$, and $w$ at each trace are $\mathsf H_q$, $\mathsf V_q$, and $\mathsf W_q$. Add the three profile contributions at their shifted traces.

In [2]:
weighted_terms = []
for label,d,trace,value in shifted:
    H_coefficient = value.subs({n2:0,w:0})
    V_coefficient = value.coefficient({n2:1})
    W_coefficient = value.coefficient({w:1})
    weighted_terms.append((label,d,trace,H_coefficient,V_coefficient,W_coefficient))
print(table([(label,trace,a,b,c) for label,d,trace,a,b,c in weighted_terms],
           header_row=['profile','trace argument','coefficient of H','coefficient of V','coefficient of W']))

def grouped_coefficient(point_count,q0,H,V,W):
    result = QQ(0)
    for label,d,trace,a,b,c in weighted_terms:
        substitutions = {q:q0,N:point_count}
        t0 = ZZ(trace.subs(substitutions))
        result += QQ(a.subs(substitutions))*H.get(t0,0)
        result += QQ(b.subs(substitutions))*V.get(t0,0)
        result += QQ(c.subs(substitutions))*W.get(t0,0)
    return result

  profile         trace argument   coefficient of H   coefficient of V   coefficient of W
├───────────────┼────────────────┼──────────────────┼──────────────────┼──────────────────┤
  split node      q - N            1/2*N^2 - 1/2      0                  1/2
  cusp            q - N + 1        N                  0                  -1
  nonsplit node   q - N + 2        -q - 1             1/2                1/2


Enumerate all squarefree degree-five and degree-six polynomials, including every nonzero leading coefficient. Character sums give $n$ and $n_2$. Infinity contributes one point in degree five and $1+\chi(\text{leading coefficient})$ in degree six. Rational roots, plus infinity in degree five, give $w$.

In [3]:
from collections import defaultdict
from itertools import product


def quadratic_character(value):
    if value == 0:
        return 0
    return 1 if value.is_square() else -1


def curve_point_count(f, field):
    Rfield.<u> = PolynomialRing(field)
    ff = Rfield(f)
    affine = field.cardinality() + sum(quadratic_character(ff(a)) for a in field)
    if ff.degree() == 5:
        infinity = 1
    else:
        infinity = 1 + quadratic_character(ff.leading_coefficient())
    return ZZ(affine + infinity)


def polynomial_tuples(field, length):
    if length == 0:
        yield []
        return
    for values in product(list(field), repeat=length):
        yield list(values)

Divide the model sums by $\#\mathrm{GL}_2(\mathbf F_q)=(q^2-1)(q^2-q)$, as in Lemma 4.5. This implementation requires prime $q>3$.

In [4]:
def model_distributions(q0):
    q0 = ZZ(q0)
    if not q0.is_prime() or q0 <= 3:
        raise ValueError('This implementation expects a prime q greater than three')
    Fq = GF(q0)
    Fq2 = GF(q0^2,'z')
    polynomial_ring.<x> = PolynomialRing(Fq)
    sums = [defaultdict(ZZ) for _ in range(3)]
    model_count = 0
    for degree in [5,6]:
        for leading in Fq:
            if leading == 0:
                continue
            for lower in polynomial_tuples(Fq,degree):
                f = leading*x^degree+sum(lower[i]*x^i for i in range(degree))
                if not f.is_squarefree():
                    continue
                model_count += 1
                points = curve_point_count(f,Fq)
                extension_points = curve_point_count(f,Fq2)
                weierstrass = len(f.roots(multiplicities=False))+ZZ(degree==5)
                trace = q0+1-points
                for accumulator,weight in zip(sums,[1,extension_points,weierstrass]):
                    accumulator[trace] += weight
    group_order = (q0^2-1)*(q0^2-q0)
    H,V,W = [{trace:QQ(value)/group_order for trace,value in accumulator.items()}
             for accumulator in sums]
    return H,V,W,model_count

The sums at $q=5,7$ contain 60,000 and 691,488 squarefree models.

In [5]:
field_sizes = [5,7]
distribution_results = {}
for q0 in field_sizes:
    H,V,W,model_count = model_distributions(q0)
    possible_points = sorted({q0+1-trace-d for trace in H for _,d,_ in profiles
                              if q0+1-trace-d>=0})
    coefficients = {point_count:grouped_coefficient(point_count,q0,H,V,W)
                    for point_count in possible_points}
    distribution_results[q0] = (H,V,W,coefficients,model_count)
    print('q =',q0,'and number of models =',model_count)
    print(table([(trace,H[trace],V[trace],W[trace]) for trace in sorted(H)],
               header_row=['trace','H_q(t)','V_q(t)','W_q(t)']))
    print(table([(point_count,value) for point_count,value in coefficients.items() if value],
               header_row=['quartic points N','weighted genus-two coefficient']))

q = 5 and number of models = 60000
  trace   H_q(t)    V_q(t)    W_q(t)
├───────┼─────────┼─────────┼────────┤
  -6      1/2       77/6      0
  -5      8/5       233/5     8/5
  -4      45/8      709/4     13/4
  -3      53/6      1687/6    21/2
  -2      33/2      526       27/2
  -1      37/2      1125/2    47/2
  0       1313/60   6613/10   203/10
  1       37/2      1125/2    47/2
  2       33/2      526       27/2
  3       53/6      1687/6    21/2
  4       45/8      709/4     13/4
  5       8/5       233/5     8/5
  6       1/2       77/6      0
  quartic points N   weighted genus-two coefficient
├──────────────────┼────────────────────────────────┤
  1                  121/24
  2                  41
  3                  581/4
  4                  1775/6
  5                  1025/2
  6                  1257/2
  7                  2873/4
  8                  584
  9                  3731/8
  10                 1355/6
  11                 205/2
  12                 41/2
  13     

Export `genus_two.json` for `weight_enumerator.ipynb`. Fields not evaluated remain uncomputed there.

In [6]:
from pathlib import Path
import json
result_dir = Path('../results')
result_dir.mkdir(exist_ok=True)
payload = {'scope':'weighted genus-two coefficients from the derived shifts and hyperelliptic-model sum',
 'fields':{str(q0):{'H':{str(t):str(v) for t,v in values[0].items()},
                    'V':{str(t):str(v) for t,v in values[1].items()},
                    'W':{str(t):str(v) for t,v in values[2].items()},
                    'coefficients':{str(z):str(v) for z,v in values[3].items()},
                    'model_count':int(values[4])}
           for q0,values in distribution_results.items()}}
(result_dir / 'genus_two.json').write_text(json.dumps(payload, indent=2) + '\n')
print('Saved', result_dir / 'genus_two.json')

Saved ../results/genus_two.json
